In [ ]:
from google.colab import drive
drive.mount('/content/week8')

In [14]:
pip install -U transformers peft trl accelerate bitsandbytes datasets

In [15]:
import torch
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer


In [16]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

TRAIN_FILE = "/content/week8/MyDrive/Week 8/llm/data/train.jsonl"
VAL_FILE   = "/content/week8/MyDrive/Week 8/llm/data/val.jsonl"

OUTPUT_DIR = "/content/week8/MyDrive/Week 8/llm/adapters"

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

LR = 2e-4
BATCH = 4
EPOCHS = 3


In [17]:
def format_sample(example):

    if example["input"].strip():
        text = (
            "### Instruction:\n"
            + example["instruction"]
            + "\n\n### Input:\n"
            + example["input"]
            + "\n\n### Response:\n"
            + example["output"]
        )
    else:
        text = (
            "### Instruction:\n"
            + example["instruction"]
            + "\n\n### Response:\n"
            + example["output"]
        )

    return {"text": text}


dataset = load_dataset(
    "json",
    data_files={
        "train": TRAIN_FILE,
        "validation": VAL_FILE
    }
)

dataset = dataset.map(
    format_sample,
    remove_columns=dataset["train"].column_names
)


In [18]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)


In [13]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [19]:
model = prepare_model_for_kbit_training(model)


In [20]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)


In [21]:
model = get_peft_model(model, lora_config)


In [22]:

model.print_trainable_parameters()


trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [23]:
model.config.use_cache = False
model.gradient_checkpointing_enable()


In [24]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,

    gradient_accumulation_steps=1,
    num_train_epochs=EPOCHS,

    learning_rate=LR,

    fp16=False,
    bf16=False,

    logging_steps=10,
    save_steps=500,
    eval_steps=500,
    eval_strategy="steps",

    save_total_limit=2,
    report_to="none"
)


In [25]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,

    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],

    formatting_func=lambda x: x["text"],

    args=training_args
)


Applying formatting function to train dataset:   0%|          | 0/108 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/108 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/108 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/108 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/11 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/11 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/11 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/11 [00:00<?, ? examples/s]

In [26]:
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss


TrainOutput(global_step=81, training_loss=0.6704397091159114, metrics={'train_runtime': 94.1211, 'train_samples_per_second': 3.442, 'train_steps_per_second': 0.861, 'total_flos': 195469489963008.0, 'train_loss': 0.6704397091159114})

In [28]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


('/content/week8/MyDrive/Week 8/llm/adapters/tokenizer_config.json',
 '/content/week8/MyDrive/Week 8/llm/adapters/chat_template.jinja',
 '/content/week8/MyDrive/Week 8/llm/adapters/tokenizer.json')

In [55]:
!cd /content/week8/MyDrive/"Week 8"/llm/utils
!python week8/MyDrive/"Week 8"/llm/utils/quantize_bnb.py


2026-02-20 13:38:44.346269: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771594724.442850   27352 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771594724.487537   27352 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771594724.561624   27352 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771594724.561671   27352 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771594724.561678   27352 computation_placer.cc:177] computation placer alr

In [58]:
ls -d llama.cpp


llama.cpp/


In [63]:
!./llama.cpp/build/bin/llama-quantize \
"/content/week8/MyDrive/Week 8/llm/quantized/model-fp16.gguf" \
"/content/week8/MyDrive/Week 8/llm/quantized/model.gguf" \
q4_0


main: build = 8117 (b908baf18)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/content/week8/MyDrive/Week 8/llm/quantized/model-fp16.gguf' to '/content/week8/MyDrive/Week 8/llm/quantized/model.gguf' as Q4_0
llama_model_loader: loaded meta data with 29 key-value pairs and 201 tensors from /content/week8/MyDrive/Week 8/llm/quantized/model-fp16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader

In [64]:
!du -sh "/content/week8/MyDrive/Week 8/llm/merged-fp16"
!du -sh "/content/week8/MyDrive/Week 8/llm/quantized/model-int8"
!du -sh "/content/week8/MyDrive/Week 8/llm/quantized/model-int4"
!ls -lh "/content/week8/MyDrive/Week 8/llm/quantized/model.gguf"


2.1G	/content/week8/MyDrive/Week 8/llm/merged-fp16
325M	/content/week8/MyDrive/Week 8/llm/quantized/model-int8
2.9G	/content/week8/MyDrive/Week 8/llm/quantized/model-int4
-rw------- 1 root root 608M Feb 20 13:58 '/content/week8/MyDrive/Week 8/llm/quantized/model.gguf'


In [1]:
from google.colab import drive

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive
